## Exploring Presidio for PII discovery and pseudonymization

[Microsoft Presidio](https://microsoft.github.io/presidio/) uses natural language processing software ([SpaCy](https://spacy.io/)) to assist in finding PII and pseudonymizing or redacting it.

In this notebook, you'll explore how to use it, where to use it and also when to not use it.

In [1]:
!python -m spacy download en_core_web_lg

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-0.1.5-py3-none-any.whl.metadata (19 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.5/116.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.5 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 22.9 MB/s eta 0:00:0031m19.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 11.5 MB/s eta 0:00:00
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install presidio_analyzer presidio_anonymizer

^C
ERROR: Operation cancelled by user


You should also install [tesseract](https://tesseract-ocr.github.io/tessdoc/Installation.html) if you want to use it with images.

In [1]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import os

ModuleNotFoundError: No module named 'presidio_analyzer'

In [23]:
text_to_pseudonymize = "His name is Mr. Samuel Jones and his phone number is +49 110 0342121345"

In [37]:
analyzer = AnalyzerEngine()
analyzer_results = analyzer.analyze(text=text_to_pseudonymize, 
                                    entities=["PHONE_NUMBER", "PERSON"], 
                                    language='en')


In [38]:
analyzer_results

[type: PERSON, start: 16, end: 28, score: 0.85,
 type: PHONE_NUMBER, start: 61, end: 71, score: 0.75]

A few more entities are [found in the full list](https://microsoft.github.io/presidio/supported_entities/).

In [19]:
supported_entities = [

    "PHONE_NUMBER",
    "US_DRIVER_LICENSE",
    "US_PASSPORT",
    "LOCATION",
    "CREDIT_CARD",
    "CRYPTO",
    "UK_NHS",
    "US_SSN",
    "US_BANK_NUMBER",
    "EMAIL_ADDRESS",
    "DATE_TIME",
    "IP_ADDRESS",
    "PERSON",
    "IBAN_CODE",
    "NRP",
    "US_ITIN",
    "MEDICAL_LICENSE",
    "URL"

]


## Practice: 
- Write a few more sentences and test out some of the other entities.
- You can also change the language. You might need to install additional packages, so [check out the SpaCy documentation](https://spacy.io/usage/models/#languages).

In [29]:
more_text = """
Yo, can you please call me back -- you have my number under JJ already. 
It's urgent because your dad is in the hospital and they are asking for his identity details. 
I htink his SS number starts with 444-22- but I forget if the last numbers are 5543 or 5544. Call me back !"""

In [35]:
more_text_results = analyzer.analyze(text=more_text, 
                                    entities=supported_entities, 
                                    language='en')

In [36]:
more_text_results

[type: PERSON, start: 1, end: 3, score: 0.85,
 type: PERSON, start: 61, end: 63, score: 0.85,
 type: DATE_TIME, start: 248, end: 252, score: 0.85]

### "Anonymizing" <- more like pseudonymizing, Engine

Now that you've found the data you want to potentially remove or protect, you can use Presidio to replace, redact, hash, mask, encrypt. Check out the [documentation](https://microsoft.github.io/presidio/anonymizer/) for information on how to use each of the choices.

Please note: the "hash" method doesn't take any salt, so please use with care (or just avoid and use encrypt instead). 

In [39]:
anonymizer = AnonymizerEngine()

anonymized_results = anonymizer.anonymize(
    text=text_to_pseudonymize,
    analyzer_results=analyzer_results,    
    operators={
        "DEFAULT": OperatorConfig("replace", 
                                  {"new_value": "<REDACTED>"}), 
        "PHONE_NUMBER": OperatorConfig("mask", {
                                        "type": "mask", 
                                        "masking_char" : "*",
                                        "chars_to_mask" : 12, 
                                        "from_end" : True}),
    }
)

In [40]:
anonymized_results

text: His name is Mr. <REDACTED> and his phone number is +49 110 **********
items:
[
    {'start': 59, 'end': 69, 'entity_type': 'PHONE_NUMBER', 'text': '**********', 'operator': 'mask'},
    {'start': 16, 'end': 26, 'entity_type': 'PERSON', 'text': '<REDACTED>', 'operator': 'replace'}
]

## Try with your text!

- I've written out a way to generate an [AES key]() because that's the only method supported by the library.
- You can also test out [other methods in the documentation](https://microsoft.github.io/presidio/anonymizer/). If it has variables listed in the table, it means you need to supply those variables and their values in the function call.
- If you run into trouble, please raise your hand!

In [85]:
def generate_aes_key(bit_length=256): 
    byte_length = int(bit_length / 8)
    return os.urandom(byte_length)

In [86]:
key = generate_aes_key()

In [87]:
key

b'U[\x12h\x07\xb0A\xde\xd5\x13o\x9e\x808\x8d\xb6\r\xfe!\xb9\x88\x82\xba\xa4\xe6\xdaO\xa5Kj,x'

In [89]:
anonymized_results = anonymizer.anonymize(
    text=more_text,
    analyzer_results=more_text_results,    
    operators={
        "DEFAULT": OperatorConfig("encrypt", 
                                  {"key": key}), 
    }
)

In [90]:
anonymized_results

text: 
Ut0r4VRhHJnSo93DNMKc1AJJZixgrZTX7mK0BZf4KzQ=, can you please call me back -- you have my number under MZes5WLPCtftjsb8nNk71n1XxX5QsEBlHgU84fjjNK8= already. 
It's urgent because your dad is in the hospital and they are asking for his identity details. 
I htink his SS number starts with 444-22- but I forget if the last numbers are WSPnsE-KrqS5suoXccmKpe5lHaQtoSNeiTj_QqcvhEE= or 5544. Call me back !
items:
[
    {'start': 332, 'end': 376, 'entity_type': 'DATE_TIME', 'text': 'WSPnsE-KrqS5suoXccmKpe5lHaQtoSNeiTj_QqcvhEE=', 'operator': 'encrypt'},
    {'start': 103, 'end': 147, 'entity_type': 'PERSON', 'text': 'MZes5WLPCtftjsb8nNk71n1XxX5QsEBlHgU84fjjNK8=', 'operator': 'encrypt'},
    {'start': 1, 'end': 45, 'entity_type': 'PERSON', 'text': 'Ut0r4VRhHJnSo93DNMKc1AJJZixgrZTX7mK0BZf4KzQ=', 'operator': 'encrypt'}
]

## Shareout

- What do you like about the library? How could you see it helping your AI/ML workflows?
- What do you wish for that it doesn't do?
- How can we imagine supporting some of those wishes?